Nama: Prieyuda Akadita S
NIM: 240401010353
Kelas: IF403

In [3]:
print("Install library.....")
!python -m pip install -q pandas numpy requests scipy

Install library.....



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
print("Library siap.")

import pandas as pd
import numpy as np
import requests
from scipy.stats.mstats import winsorize

# --- BAGIAN 1: DATA CLEANING (Langkah 3-10) ---

# Langkah 3: Muat dataset dari Google Drive (Direct Link)
# Tips: Menggunakan link ekspor agar pandas bisa langsung membaca CSV
file_id = '1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'
url = f'https://drive.google.com/uc?id={file_id}'

df = pd.read_csv(url)

# Langkah 4: Eksplorasi awal
print("--- Eksplorasi Awal ---")
df.info()
print("\nStatistik Deskriptif:\n", df.describe())
print("\nJumlah Missing Values:\n", df.isnull().sum())

# Langkah 5: Hapus baris duplikat
df.drop_duplicates(inplace=True)

# Langkah 6: Normalisasi string
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Langkah 7: Imputasi missing values
# Median untuk numerik
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['tahun_bangun'] = df['tahun_bangun'].fillna(df['tahun_bangun'].median())

# Modus untuk kategorik (kamar & kondisi jika ada yang kosong)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
df['kondisi'] = df['kondisi'].fillna(df['kondisi'].mode()[0])

# Langkah 8: Tangani outlier dengan IQR Fence
for col in ['harga_juta', 'luas_m2']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower_fence, upper_fence)

# Langkah 9: Validasi akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print("\nValidasi Berhasil: Tidak ada missing value atau duplikat.")

# Langkah 10: Ekspor ke CSV
df.to_csv('housing_clean.csv', index=False)
print("File 'housing_clean.csv' berhasil disimpan!")


# --- BAGIAN 2: AKSES API (Langkah 11) ---

# Langkah 11: Akses API JSONPlaceholder dan simpan sebagai DataFrame
print("\n--- Mengambil Data dari API ---")
api_url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(api_url)

if response.status_code == 200:
    data_api = response.json()
    df_api = pd.DataFrame(data_api)
    print("Berhasil mengambil data dari API.")
    print(df_api.head()) # Menampilkan 5 data pertama dari API
else:
    print("Gagal mengambil data dari API.")

Library siap.
--- Eksplorasi Awal ---
<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    str    
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    str    
dtypes: float64(3), int64(2), str(2)
memory usage: 7.2 KB

Statistik Deskriptif:
                id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02 

Belajar untuk data cleaning karena di kenyataan data ideal yang sudah bersih jarang ditemui